### Initializing (imports, settings, loading rec, clalc conc)

In [ ]:
%matplotlib qt

import importlib
import pandas as pd
import xarray as xr
import cedalion
from cedalion.io.snirf import read_snirf
from cedalion.nirs.cw import int2od
from cedalion.nirs.cw import od2conc
# import matplotlib.pyplot as plt

pd.set_option('display.max_rows', 10)
xr.set_options(display_expand_data=False)

def calc_concentratoin(rec):
    od = int2od(rec["amp"])
    dpf = xr.DataArray([6, 6], dims="wavelength", coords={"wavelength" : od.wavelength})
    return od2conc(od, rec.geo3d, dpf)


In [ ]:
path_to_snirf_file = r"D:\WorkPy_Bench\03_Cedalion\2025-05-03_001\2025-05-03_001_cut.snirf"  # noqa: E501

recordings = read_snirf(path_to_snirf_file)
rec = recordings[0]
rec["conc"] = calc_concentratoin(rec)

In [ ]:
display(rec["conc"].sel(chromo="HbO"))

### Creation of Bvp_Container --> BVP extraction --> Waveform: extraction

In [ ]:
channel_to_plot = 'S1D15'

In [ ]:
# importlib.reload(cedalion.sigproc.bvp_wav_ana_v12)
# importlib.reload(cedalion.sigproc.bvp_container)

In [ ]:
from cedalion.dataclasses.bvp_container import BVP_Container
bvp_cont = BVP_Container()
bvp_cont

In [ ]:
from cedalion.sigproc.bvp_wav_ana_v12 import extract_bvp
bvp_ts = extract_bvp(rec["conc"].sel(chromo="HbO"))
bvp_cont['bvp_ts'] = bvp_ts

In [ ]:
print(bvp_cont)
display(bvp_cont['bvp_ts'])

In [ ]:
from cedalion.sigproc.bvp_wav_ana_v12 import extract_waveforms
wav_storage_user, wav_storage_details = extract_waveforms(bvp_ts.sel(compound="bvp_ts"))
bvp_cont.wav_storage_user = wav_storage_user
bvp_cont.wav_storage_details = wav_storage_details

In [ ]:
print(bvp_cont)

In [ ]:
# from cedalion.sigproc.bvp_wav_ana_v12 import plot_wavs_4x
# plot_wavs_4x(bvp_cont, channel_to_plot)

In [ ]:
# from cedalion.sigproc.bvp_wav_ana_v12 import plot_concts_bvpts
# plot_concts_bvpts(bvp_cont, channel_to_plot)

### Waveforms: artefact removal

In [ ]:
# importlib.reload(cedalion.sigproc.bvp_wav_ana_v12)

In [ ]:
from cedalion.sigproc.bvp_wav_ana_v12 import remove_artifact_waveforms
bvp_cont.wav_storage_user, bvp_cont.wav_storage_details = remove_artifact_waveforms(
    bvp_cont['bvp_ts'],
    bvp_cont.wav_storage_user,
    bvp_cont.wav_storage_details
)

In [ ]:
print(bvp_cont)

In [ ]:
# from cedalion.sigproc.bvp_wav_ana_v12 import plot_wavs_woa
# plot_wavs_woa(bvp_cont, channel_to_plot)

### Waveforms: classification

In [ ]:
importlib.reload(cedalion.sigproc.bvp_wav_ana_v12)

In [ ]:
from cedalion.sigproc.bvp_wav_ana_v12 import classify_waveforms
bvp_cont.wav_storage_user, bvp_cont.wav_storage_details = classify_waveforms(
    bvp_cont, 'delta')
bvp_cont.wav_storage_user, bvp_cont.wav_storage_details = classify_waveforms(
    bvp_cont, 'max')

for ch in bvp_cont['bvp_ts'].channel.values:
    print(wav_storage_details[ch]["text_num_del_wavs"])

In [ ]:
print(bvp_cont)

In [ ]:
# from cedalion.sigproc.bvp_wav_ana_v12 import plot_wavs_classes
# plot_wavs_classes(bvp_cont, channel_to_plot, 'max')

### BVPA extraction

In [ ]:
# importlib.reload(cedalion.sigproc.bvp_wav_ana_v12)

In [ ]:
from cedalion.sigproc.bvp_wav_ana_v12 import extract_bvpa
bvpa_ts = extract_bvpa(
    bvp_cont['bvp_ts'].sel(compound="bvp_ts"),
    bvp_cont.wav_storage_user
)
bvp_cont['bvpa_ts'] = bvpa_ts

In [ ]:
print(bvp_cont)
display(bvp_cont['bvpa_ts'])

In [ ]:
# from cedalion.sigproc.bvp_wav_ana_v12 import plot_bvpts_bvpats
# plot_bvpts_bvpats(bvp_cont, channel_to_plot)

### Pulse Rate: extraction

In [ ]:
# importlib.reload(cedalion.sigproc.bvp_wav_ana_v12)

In [ ]:
from cedalion.sigproc.bvp_wav_ana_v12 import extract_pulse_rate
pulse_rate_ts = extract_pulse_rate(
    bvp_cont['bvp_ts'].sel(compound="bvp_ts"),
    bvp_cont.wav_storage_user
)
bvp_cont['pulse_rate_ts'] = pulse_rate_ts

In [ ]:
print(bvp_cont)
display(bvp_cont['pulse_rate_ts'])

In [ ]:
# from cedalion.sigproc.bvp_wav_ana_v12 import plot_pulse_rate
# plot_pulse_rate(bvp_cont, channel_to_plot)

### Pulse Rate: filtering

In [ ]:
# importlib.reload(cedalion.sigproc.bvp_wav_ana_v12)

In [ ]:
from cedalion.sigproc.bvp_wav_ana_v12 import filter_pulse_rate
pulse_rate_ts = filter_pulse_rate(
    pulse_rate_ts,
    fmin=0.1, fmax=0.45,
    butter_order=2)
bvp_cont['pulse_rate_ts'] = pulse_rate_ts

In [ ]:
print(bvp_cont)
display(bvp_cont['pulse_rate_ts'])

In [ ]:
from cedalion.sigproc.bvp_wav_ana_v12 import plot_bvpa_pr_comparison
plot_bvpa_pr_comparison(bvp_cont, channel_to_plot)

### comprehensive Plot

In [ ]:
# importlib.reload(cedalion.sigproc.bvp_wav_ana_v12)

In [ ]:
# from cedalion.sigproc.bvp_wav_ana_v12 import plot_concts_bvpats_pr
# plot_concts_bvpats_pr(bvp_cont, channel_to_plot)

### Wavelet Coherence Analysis: BVPA - LPR

In [ ]:
# importlib.reload(cedalion.sigproc.bvp_wav_ana_pycwt)

In [ ]:
# importlib.reload(cedalion.sigproc.bvp_wav_ana_pycwt)
from cedalion.sigproc.bvp_wav_ana_pycwt import wavelet_coherence_plot  # noqa: E402

ch = 'S10D9'

fs = 50
dt = 1.0 / fs

x = bvp_cont['bvpa_ts'].sel(compound='bvpa_smooth', channel=ch).to_numpy()
# x = x[0:10000]
y = bvp_cont['pulse_rate_ts'].sel(compound='pulse_rate_smooth',
                                  channel=ch).to_numpy()
# y = y[0:10000]

WCT, aWCT, freq, t, coi = wavelet_coherence_plot(
                        x, y, dt=dt, dj=1/12, s0=0.5, J=100,
                        do_zscore=True, do_detrend=True, do_sig=True,
                        wavelet_name="morlet",
                        coherence_thresh=0.9,
                        arrow_step_time=50*30,
                        arrow_step_scale=4)